In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/iterative_9_10_not_end/"

In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*02:01']

# 2. Train HMMS

In [4]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions


NUM_RUNS = 2
TARGET_LENGTHS= [9, 10]

In [5]:
start_length = 1

## 2.1 Filter data

In [6]:
per_allele_per_kfold_per_length_binders_train_filtered = dict()
per_allele_per_kfold_per_length_binders_test_filtered = dict()


for allele_name in ALLELES:
    per_allele_per_kfold_per_length_binders_train_filtered[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_filtered[allele_name] = dict()

    for split_num in per_allele_per_kfold_per_length_binders_train[allele_name].keys():
        per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num] = dict()

        for target_length in per_allele_per_kfold_per_length_binders_train[allele_name][split_num].keys():
            data_array = per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]
            per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num][target_length] = np.array([s[:start_length] for s in data_array ])

            data_array = per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]
            per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num][target_length] = np.array([s[:start_length] for s in data_array ])

## 2.2 Training process

In [7]:
stop_threshold = 0
cutoff = 0.00005
per_symbol_portion = -22 / 9
for target_length in TARGET_LENGTHS:
    target_length_for_training = start_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

1.9212111111111116

In [8]:
%%time
algorithm = 'viterbi'

per_name_models = dict()
per_allele_per_run_per_split_models=dict()
for allele_name in ALLELES:
    print(f"allele {allele_name}")
    per_allele_per_run_per_split_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"run {run_num}")

        print()


        per_kfold_per_length_data =  per_allele_per_kfold_per_length_binders_train_filtered[allele_name]

        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test_filtered[allele_name],
                                        verbose=True,
                                        multiple_check_input=False,
                                        algorithm=algorithm,
                                        name_prefix=f"{run_num}_run",
                                        num_states=4,
                                        n_jobs=10,
                                        stop_threshold=stop_threshold,
                                        min_iterations=5
                                        )
        per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


allele HLA-A*02:01
run 0

split 0
train size 31438
Shuffling for fairness 
Simple training: Train Length 31438, init_batch_size None
[1] Improvement: 53945.13277573105	Time (s): 3.422
[2] Improvement: 0.0	Time (s): 2.719
[3] Improvement: 0.0	Time (s): 2.766
[4] Improvement: 0.0	Time (s): 2.734
[5] Improvement: 0.0	Time (s): 2.766
Total Training Improvement: 53945.13277573105
Total Training Time (s): 17.4843
run 1

split 0
train size 31438
Shuffling for fairness 
Simple training: Train Length 31438, init_batch_size None
[1] Improvement: 56302.47946986584	Time (s): 2.734
[2] Improvement: 0.0	Time (s): 2.719
[3] Improvement: 0.0	Time (s): 2.781
[4] Improvement: 0.0	Time (s): 2.734
[5] Improvement: 0.0	Time (s): 2.781
Total Training Improvement: 56302.47946986584
Total Training Time (s): 16.5781
Wall time: 34.4 s


In [10]:
from hmm_logic_methods import add_more_states_and_reset_transitions, add_more_states_to_the_end

In [11]:
per_allele_per_run_per_split_prepared_models = dict()
per_allele_per_run_per_split_original_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    per_allele_per_run_per_split_original_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        per_allele_per_run_per_split_original_models[allele_name][run_num] = dict()
        for split_num, model in per_allele_per_run_per_split_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_iterations"
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model
            per_allele_per_run_per_split_original_models[allele_name][run_num][split_num] = prepared_model

In [12]:
initial_length_for_training = start_length
max_length = max(TARGET_LENGTHS)

In [13]:
ADD_STATES_NUM = 4
for iteration in range(initial_length_for_training + 1,max_length+1):
    print(f"To length {iteration}")
    source_models = per_allele_per_run_per_split_prepared_models

    # copy models and rename
    per_allele_per_run_per_split_prepared_models = dict()
    for allele_name in ALLELES:
        print(f"allele {allele_name}")
        per_allele_per_run_per_split_prepared_models[allele_name] = dict()
        for run_num in range(NUM_RUNS):
            print(f"run {run_num}")
            per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
            for split_num, model in source_models[allele_name][run_num].items():
                print(f"split {split_num}")
                prepared_model = model.copy()
                old_states_names = [state.name for state in prepared_model.states]
                new_states_names =  state_names = ["s{:03d}".format(i) for i in range(len(prepared_model.states),len(prepared_model.states) + ADD_STATES_NUM)]
                original_model = per_allele_per_run_per_split_original_models[allele_name][run_num][split_num]
                new_model = add_more_states_and_reset_transitions(prepared_model,
                                                                  ADD_STATES_NUM,
                                                                  new_sates_names =new_states_names,
                                                                  new_name = original_model.name + f"_to_length_{iteration}",
                                                                  new_probs="fair")

                print("Freezing the distribution for old states")
                print(f"old state {old_states_names}")
                frozen = []
                for state in new_model.states:
                    if (state.name in old_states_names) and state is not new_model.start and state is not new_model.end:
                        state.distribution.freeze()
                        frozen.append(state.name)
                print(f"freezed {frozen}")

                per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = new_model



    # make new data + 1 length
    per_allele_per_kfold_per_length_binders_train_filtered = dict()
    per_allele_per_kfold_per_length_binders_test_filtered = dict()

    for allele_name in ALLELES:
        per_allele_per_kfold_per_length_binders_train_filtered[allele_name] = dict()
        per_allele_per_kfold_per_length_binders_test_filtered[allele_name] = dict()

        for split_num in per_allele_per_kfold_per_length_binders_train[allele_name].keys():
            per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num] = dict()
            per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num] = dict()

            for target_length in per_allele_per_kfold_per_length_binders_train[allele_name][split_num].keys():
                data_array = per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]
                per_allele_per_kfold_per_length_binders_train_filtered[allele_name][split_num][target_length] = np.array([s[:iteration] for s in data_array ])

                data_array = per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]
                per_allele_per_kfold_per_length_binders_test_filtered[allele_name][split_num][target_length] = np.array([s[:iteration] for s in data_array ])
    # #train using new data
    # stop criteria correction
    for target_length in TARGET_LENGTHS:
        target_length_for_training = iteration
        train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
        stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
    stop_threshold = stop_threshold / len(TARGET_LENGTHS)

    per_allele_per_run_per_split_new_histories = dict()
    for allele_name in ALLELES:
        per_allele_per_run_per_split_new_histories[allele_name] = dict()
        for run_num in range(NUM_RUNS):
            per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train_filtered[allele_name]
            models, histories = train_model(allele_name,
                                            per_kfold_per_length_data,
                                            TARGET_LENGTHS,
                                            multiple_check_input=False,
                                            algorithm=algorithm,
                                            verbose=True,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test_filtered[allele_name],
                                            stop_threshold=stop_threshold,
                                            min_iterations= 5,
                                            prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num])
            per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
            per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
            # per_allele_per_run_per_split_models[allele_name][run_num] = models
            for split_num, model in models.items():
                per_name_models[model.name] = model
            # save models
            for split_num, model in models.items():
                history = histories[split_num]
                save_model(TARGET_PATH_TO_RESULTS, model,history)


To length 2
allele HLA-A*02:01
run 0
split 0
Initial length 6
num states to add 4
After length 10
 N is  8
sum from start 2.0000000000000004
Freezing the distribution for old states
old state ['s000', 's001', 's002', 's003', '0_run-9-10_model-start', '0_run-9-10_model-end']
freezed ['s000', 's001', 's002', 's003']
run 1
split 0
Initial length 6
num states to add 4
After length 10
 N is  8
sum from start 2.0000000000000004
Freezing the distribution for old states
old state ['s000', 's001', 's002', 's003', '1_run-9-10_model-start', '1_run-9-10_model-end']
freezed ['s000', 's001', 's002', 's003']
split 0
train size 31438
Shuffling for fairness 
Simple training: Train Length 31438, init_batch_size None
[1] Improvement: -24704.904924658418	Time (s): 1.5
[2] Improvement: 0.0	Time (s): 1.594
[3] Improvement: 0.0	Time (s): 1.625
[4] Improvement: 0.0	Time (s): 1.531
[5] Improvement: 0.0	Time (s): 1.547
Total Training Improvement: -24704.904924658418
Total Training Time (s): 9.4531
split 0
train

In [14]:
per_name_models.keys()

dict_keys(['0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0', '1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0', '0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_2', '1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_2', '0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_3', '1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_3', '0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_4', '1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_4', '0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_5', '1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_5', '0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_6', '1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_6', '0_

In [15]:
from hmm_visualization_methods import *

In [16]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
Wall time: 9.94 s


In [17]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
2
3
4
5
6
7
in validate_matrix(): Row sums in df are not close to 1. Reormalizing rows...
8
9
in validate_matrix(): Row sums in df are not close to 1. Reormalizing rows...
10
11
in validate_matrix(): Row sums in df are not close to 1. Reormalizing rows...
12
13
in validate_matrix(): Row sums in df are not close to 1. Reormalizing rows...
14
15
in validate_matrix(): Row sums in df are not close to 1. Reormalizing rows...
16
17
in validate_matrix(): Row sums in df are not close to 1. Reormalizing rows...
18
19
Wall time: 1min 54s


In [18]:
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0) (Delete zero edges)
Converting graph for model (1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0) (Delete zero edges)
Converting graph for model (0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_2) (Delete zero edges)
Converting graph for model (1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_2) (Delete zero edges)
Converting graph for model (0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_3) (Delete zero edges)
Converting graph for model (1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_3) (Delete zero edges)
Converting graph for model (0_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_4) (Delete zero edges)
Converting graph for model (1_run-9-10-4-st-viterbi-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_4) (Delete ze

## Aux viterbi

In [20]:
target_length_for_training = 9
total_length = 9
cutoff = 0.00005
train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9])
## Stop threshold Hardcoded here
stop_threshold =  abs(cutoff * train_size * -22/ total_length  * target_length_for_training)
per_allele_per_run_per_split_new_histories = dict()

for allele_name in ALLELES:
    for run_num in range(NUM_RUNS):
        for split_num, model in per_allele_per_run_per_split_prepared_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_plus_viterbi"

            for state in prepared_model.states:
                if state is not prepared_model.start and state is not prepared_model.end:
                    state.distribution.thaw()
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

In [21]:
per_name_new_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train_filtered[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        multiple_check_input=False,
                                        algorithm='viterbi',
                                        verbose=True,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test_filtered[allele_name],
                                        stop_threshold=stop_threshold,
                                        min_iterations= 5,
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num])
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_new_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

split 0
train size 31438
Shuffling for fairness 
Simple training: Train Length 31438, init_batch_size None
[1] Improvement: 13333.494434460416	Time (s): 4.31
[2] Improvement: 1476.1219541991595	Time (s): 4.211
[3] Improvement: 1091.4378207712434	Time (s): 4.288
[4] Improvement: 563.2459265096113	Time (s): 4.243
[5] Improvement: 280.17230245273095	Time (s): 4.23
[6] Improvement: -13.655568758957088	Time (s): 4.28
Total Training Improvement: 16730.816869634204
Total Training Time (s): 34.1960
split 0
train size 31438
Shuffling for fairness 
Simple training: Train Length 31438, init_batch_size None
[1] Improvement: 7346.127731366665	Time (s): 4.22
[2] Improvement: 1169.423184562358	Time (s): 4.553
[3] Improvement: 734.1462855885038	Time (s): 4.444
[4] Improvement: 427.19725030031987	Time (s): 4.28
[5] Improvement: 569.2957734644879	Time (s): 4.296
[6] Improvement: 851.0476745212218	Time (s): 4.226
[7] Improvement: 602.516266839928	Time (s): 4.231
[8] Improvement: 219.98852605745196	Time (

In [22]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 567 ms


In [23]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 35.4 s


In [24]:
per_name_pyviz = dict()
for name, model in per_name_new_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-10-4-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_10_plus_viterbi) (Delete zero edges)
new starting point
new starting point
new starting point
new starting point
new starting point
Converting graph for model (1_run-9-10-4-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_10_plus_viterbi) (Delete zero edges)
new starting point
new starting point


In [48]:
model

{
    "class" : "HiddenMarkovModel",
    "name" : "1_run-9-10-4-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_10_plus_viterbi",
    "start" : {
        "class" : "State",
        "distribution" : null,
        "name" : "1_run-9-10-4-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_10-start",
        "weight" : 1.0
    },
    "end" : {
        "class" : "State",
        "distribution" : null,
        "name" : "1_run-9-10-4-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_iterations_to_length_10-end",
        "weight" : 1.0
    },
    "states" : [
        {
            "class" : "State",
            "distribution" : {
                "class" : "Distribution",
                "dtype" : "str",
                "name" : "DiscreteDistribution",
                "parameters" : [
                    {
                        "A" : 0.0,
                        "C" : 0.0,
                        "D" : 0.3798449612403101,
                        "E" :

In [24]:
model.copy().dense_transition_matrix()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ],
       [0.29362555, 0.14056238, 0.07560914, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [24]:
model.states[37]

{
    "class" : "State",
    "distribution" : {
        "class" : "Distribution",
        "dtype" : "str",
        "name" : "DiscreteDistribution",
        "parameters" : [
            {
                "H" : 0.007322012774575479,
                "G" : 0.0,
                "I" : 0.0,
                "A" : 0.22152983330736875,
                "Q" : 0.0,
                "Y" : 0.05172145193955445,
                "L" : 0.15485278080697928,
                "S" : 0.0,
                "P" : 0.0,
                "E" : 0.0,
                "R" : 0.0,
                "M" : 0.0,
                "T" : 0.0,
                "W" : 0.0022589188347094565,
                "F" : 0.043231032871163734,
                "V" : 0.5190839694656488,
                "K" : 0.0,
                "D" : 0.0,
                "C" : 0.0,
                "N" : 0.0
            }
        ],
        "frozen" : false
    },
    "name" : "s039",
    "weight" : 1.0
}